# Customer Segmentation with K-Means
## Unsupervised Machine Learning Project

**Objective:** Group customers into meaningful segments using K-Means clustering and explain each segment in business language.

**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn, scikit-learn  
**Level:** Beginner–Intermediate  
**Dataset:** Synthetic customer shopping data generated in this notebook (no upload required).

### Project workflow
1. Generate and inspect customer data.
2. Prepare and scale features for clustering.
3. Choose a suitable number of clusters using the Elbow Method and Silhouette Score.
4. Train the K-Means model.
5. Profile each segment and describe it in business terms.
6. Visualize clusters and export the segmented customer list.

**Important:** K-Means discovers patterns based on the selected features. The business labels are interpretations, not ground-truth customer types.

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

print("Libraries imported successfully.")

## 2. Create a sample customer dataset

Each row represents one customer. We generate three broad shopping patterns to make the clustering exercise interpretable. In a real project, replace this synthetic dataset with approved customer transaction data.

Features:
- `age`: customer age in years
- `annual_income_k`: annual income in thousands of currency units
- `spending_score`: shopping engagement score from 1 to 100
- `annual_purchases`: number of purchases in a year
- `avg_order_value`: average amount spent per order

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# Synthetic customer profiles used only to generate realistic practice data
n_each = 100
profiles = [
    # Higher income, higher spending, frequent purchases
    {"age": (25, 48), "income": (65, 130), "score": (70, 100), "purchases": (18, 42), "aov": (120, 350)},
    # Higher income, lower spending, fewer purchases
    {"age": (30, 65), "income": (70, 150), "score": (10, 42), "purchases": (3, 15), "aov": (80, 250)},
    # Lower-to-mid income, moderate spending and purchase frequency
    {"age": (18, 60), "income": (20, 75), "score": (35, 75), "purchases": (8, 25), "aov": (25, 140)}
]

rows = []
customer_id = 1
for profile in profiles:
    for _ in range(n_each):
        rows.append({
            "customer_id": customer_id,
            "age": rng.integers(*profile["age"]),
            "annual_income_k": rng.uniform(*profile["income"]),
            "spending_score": rng.uniform(*profile["score"]),
            "annual_purchases": rng.integers(*profile["purchases"]),
            "avg_order_value": rng.uniform(*profile["aov"])
        })
        customer_id += 1

customers = pd.DataFrame(rows)
customers[["annual_income_k", "spending_score", "avg_order_value"]] = customers[
    ["annual_income_k", "spending_score", "avg_order_value"]
].round(2)

print("Dataset shape:", customers.shape)
display(customers.head())
display(customers.describe().round(2))

## 3. Explore the dataset

Check column types, missing values, duplicate customer IDs, and feature distributions before modelling.

In [ ]:
print("Data types and non-null counts:")
customers.info()

print("\\nMissing values:")
display(customers.isna().sum().to_frame("missing_count"))

print("\\nDuplicate customer IDs:", customers["customer_id"].duplicated().sum())

features = ["age", "annual_income_k", "spending_score", "annual_purchases", "avg_order_value"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for col, ax in zip(features, axes.flat):
    sns.histplot(customers[col], kde=True, ax=ax)
    ax.set_title(col.replace("_", " ").title())
for ax in axes.flat[len(features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Prepare features and scale them

**Decision:** Exclude `customer_id` because it is only an identifier and carries no customer behaviour meaning. Use the five numeric behaviour/demographic features listed below.

**Why scaling matters:** Income and order value have much larger numeric ranges than scores or purchase counts. K-Means uses distance, so standardization prevents large-scale variables from dominating the clusters.

For this synthetic dataset there are no missing values. For real data, decide how to handle missing values before scaling.

In [ ]:
X = customers[features].copy()

assert X.isna().sum().sum() == 0, "Handle missing values before fitting K-Means."

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features used:", features)
print("Scaled matrix shape:", X_scaled.shape)

## 5. Choose the number of clusters (K)

Use two diagnostics:
- **Elbow Method:** look for a point where the reduction in within-cluster sum of squares (inertia) starts slowing down.
- **Silhouette Score:** measures how well-separated and cohesive the clusters are. Higher is generally better; scores near 1 indicate well-separated clusters, around 0 indicate overlap, and negative values can indicate poor assignments.

These are decision aids, not automatic truth. Also consider whether the resulting groups are understandable and useful for the business.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_values), silhouette_scores, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

score_table = pd.DataFrame({"k": list(k_values), "inertia": inertias, "silhouette_score": silhouette_scores})
display(score_table.round(3))

## 6. Train the final K-Means model

For this practice dataset, use **K = 3** because the data was generated with three broad shopping patterns. In a real project, select K after reviewing the elbow plot, silhouette scores, segment stability, and business usefulness.

K-Means assigns numeric labels (0, 1, 2). These labels do not indicate rank or value.

In [ ]:
FINAL_K = 3

kmeans = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
customers["cluster"] = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(X_scaled, customers["cluster"])
print(f"Model trained with K={FINAL_K}")
print(f"Silhouette score: {final_silhouette:.3f}")
display(customers[["customer_id", "cluster"]].head(10))

## 7. Profile each customer segment

Calculate the average and median feature values for each cluster. These summaries help explain how groups differ.

In [ ]:
cluster_profile = customers.groupby("cluster")[features].agg(["mean", "median", "count"])
display(cluster_profile.round(2))

cluster_means = customers.groupby("cluster")[features].mean().round(2)
display(cluster_means)

## 8. Describe the segments in business language

The following labels are generated from the observed cluster averages using transparent, simple rules. They are example business interpretations and should be reviewed with domain experts before using them in campaigns.

The rule order matters: a segment can meet more than one condition, so the first matching description is used.

In [ ]:
def describe_segment(row):
    if row["spending_score"] >= 65 and row["annual_purchases"] >= 18:
        return "High-value frequent shoppers"
    elif row["annual_income_k"] >= 65 and row["spending_score"] < 45:
        return "Affluent but cautious shoppers"
    elif row["annual_income_k"] < 65 and row["spending_score"] >= 35:
        return "Mainstream / developing customers"
    else:
        return "Mixed-behaviour customers"

business_labels = {
    "High-value frequent shoppers": (
        "Shop frequently and show strong spending engagement.",
        "Reward loyalty, offer early access, and test premium bundles."
    ),
    "Affluent but cautious shoppers": (
        "Have relatively high income but lower shopping engagement.",
        "Use personalized recommendations, quality messaging, and targeted re-engagement."
    ),
    "Mainstream / developing customers": (
        "Represent a broad value-conscious group with moderate engagement.",
        "Use relevant offers, cross-sells, and simple loyalty incentives."
    ),
    "Mixed-behaviour customers": (
        "Do not fit the other simple profiles clearly.",
        "Study their purchase journeys and test smaller targeted campaigns."
    )
}

cluster_label_map = {}
for cluster_id, row in cluster_means.iterrows():
    cluster_label_map[cluster_id] = describe_segment(row)

customers["segment"] = customers["cluster"].map(cluster_label_map)

segment_summary = customers.groupby(["cluster", "segment"]).agg(
    customer_count=("customer_id", "count"),
    avg_age=("age", "mean"),
    avg_income_k=("annual_income_k", "mean"),
    avg_spending_score=("spending_score", "mean"),
    avg_annual_purchases=("annual_purchases", "mean"),
    avg_order_value=("avg_order_value", "mean")
).round(2).reset_index()

display(segment_summary)

## 9. Visualize the clusters

First, plot income against spending score. Then use PCA to project all five standardized features into two dimensions for a compact visual overview. PCA is only a visualization step; the K-Means model was trained on all five scaled features.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=customers,
    x="annual_income_k",
    y="spending_score",
    hue="segment",
    palette="Set2",
    s=65
)
plt.title("Customer Segments: Income vs Spending Score")
plt.xlabel("Annual income (thousands)")
plt.ylabel("Spending score")
plt.legend(title="Business segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
plot_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "segment": customers["segment"]
})

plt.figure(figsize=(9, 6))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="segment", palette="Set2", s=65)
plt.title("Customer Segments: PCA Projection")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Business segment", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"Variance explained by 2 PCA components: {pca.explained_variance_ratio_.sum():.1%}")

## 10. Compare segment sizes and business metrics

In [ ]:
segment_counts = customers["segment"].value_counts().rename_axis("segment").reset_index(name="customers")
segment_counts["percentage"] = (segment_counts["customers"] / len(customers) * 100).round(1)
display(segment_counts)

plt.figure(figsize=(9, 4))
sns.barplot(data=segment_counts, x="segment", y="customers")
plt.title("Number of Customers in Each Segment")
plt.xlabel("")
plt.ylabel("Customer count")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.barplot(data=segment_summary, x="segment", y="avg_annual_purchases")
plt.title("Average Annual Purchases by Segment")
plt.xlabel("")
plt.ylabel("Average annual purchases")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 11. Business recommendations

Use the profile table and the following recommendations as a starting point:

| Segment | Business interpretation | Suggested action |
|---|---|---|
| High-value frequent shoppers | Highly engaged customers with frequent purchases | Loyalty rewards, VIP experiences, premium bundles |
| Affluent but cautious shoppers | Higher income but lower engagement | Personalized offers, product education, re-engagement |
| Mainstream / developing customers | Moderate engagement and broad value needs | Cross-sell, accessible promotions, loyalty incentives |
| Mixed-behaviour customers | Does not fit a simple profile | Investigate customer journeys and test targeted campaigns |

**Do not assume** that a segment is profitable solely from income or spending score. To estimate profitability, add reliable margin, returns, discount, and customer lifetime value data.

## 12. Export the segmented customer dataset and summary

The notebook saves:
- `customer_segments.csv` — every customer with a cluster number and business segment.
- `segment_summary.csv` — aggregate statistics for each segment.

In [ ]:
output_dir = Path("/content") if Path("/content").exists() else Path(".")
customer_file = output_dir / "customer_segments.csv"
summary_file = output_dir / "segment_summary.csv"

customers.to_csv(customer_file, index=False)
segment_summary.to_csv(summary_file, index=False)

print("Files saved:")
print(customer_file)
print(summary_file)

try:
    from google.colab import files
    files.download(str(customer_file))
    files.download(str(summary_file))
except ImportError:
    print("Outside Google Colab: files are saved in the current working directory.")

## 13. Project conclusion

In this project, K-Means grouped customers using age, annual income, spending score, purchase frequency, and average order value.

You:
- explored and prepared a customer dataset,
- standardized features so distance calculations are fair,
- reviewed candidate cluster counts with the Elbow Method and Silhouette Score,
- trained a K-Means model,
- profiled and visualized the resulting segments,
- translated cluster patterns into business descriptions and actions,
- exported customer-level and segment-level CSV files.

### Limitations and next steps
- The dataset is synthetic; results will differ on real customer data.
- K-Means works best with numeric features and can be affected by outliers and feature selection.
- Segment labels are interpretations, not predictions or verified customer identities.
- For production use, test stability over time, review fairness/privacy, and validate campaign impact with experiments.